# Notebook 02 - Fine-Tuning de LLM para Dominio Medico

## Tech Challenge Fase 3 - Assistente Virtual Medico

Este notebook cobre:
1. Configuracao do pipeline de fine-tuning
2. Preparacao dos dados para treinamento
3. Fine-tuning com LoRA/QLoRA (eficiente em memoria)
4. Avaliacao do modelo treinado
5. Exportacao do modelo

---
## 1. Conceitos de Fine-Tuning

O fine-tuning consiste em adaptar um modelo LLM generico (LLaMA, Falcon, etc.) a um dominio especifico, utilizando dados proprios do hospital.

### Por que Fine-Tuning?
- O modelo generico nao conhece protocolos especificos do hospital
- Fine-tuning permite personalizar o modelo para o contexto medico
- Resultados mais precisos e consistentes em relacao ao dominio

### Tecnicas Utilizadas:
- **LoRA (Low-Rank Adaptation)**: Ajusta apenas pesos de baixa rank, economizando memoria
- **QLoRA**: LoRA com quantizacao 4-bit, permitindo treinamento em GPUs menores
- **Prompt Tuning**: Ajusta apenas os embeddings do prompt

---
## 2. Instalacao e Configuracao

In [1]:
# Instalacao de dependencias para fine-tuning
!pip install transformers datasets accelerate peft bitsandbytes trl -q

In [2]:
import os
import json
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from dotenv import load_dotenv

load_dotenv()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

/Users/rodrigofranco/Environment/FIAP/fase 3/aulas/projeto-assistente-medico/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.14.0
CUDA available: False


---
## 3. Carregamento do Dataset

In [3]:
# Carregar dados preparados (dataset com618 exemplos)
dataset_file = "../data/dataset_alpaca.json"

with open(dataset_file, 'r', encoding='utf-8') as f:
    dados_treino = json.load(f)

# Garantir que todos os registros tenham o campo 'text'
for item in dados_treino:
    if 'text' not in item:
        instruction = item.get('instruction', '')
        inp = item.get('input', '')
        output = item.get('output', '')
        item['text'] = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n{output}"

print(f"Total de exemplos: {len(dados_treino)}")
print(f"\nFontes dos dados:")

# Contar por fonte
fontes = {}
for item in dados_treino:
    fonte = item.get('source', 'desconhecida')
    fontes[fonte] = fontes.get(fonte, 0) + 1

for fonte, count in fontes.items():
    print(f"  - {fonte}: {count} exemplos")

print(f"\nExemplo formatado:")
print(dados_treino[0]['text'][:500])

# Converter para HuggingFace Dataset
dataset = Dataset.from_list(dados_treino)
dataset = dataset.train_test_split(test_size=0.3, seed=42)

print(f"\nDataset de treino: {len(dataset['train'])} exemplos")
print(f"Dataset de validacao: {len(dataset['test'])} exemplos")

Total de exemplos: 11018

Fontes dos dados:
  - synthetic: 18 exemplos
  - pubmedqa: 1000 exemplos
  - medquad: 10000 exemplos

Exemplo formatado:
### Instruction:
Voce e um assistente medico especializado em Pneumologia. Responda sobre o protocolo de pneumonia hospitalar.

### Input:
Qual o protocolo de tratamento para pneumonia hospitalar adquirida em ambiente de UTI?

### Response:
Protocolo de Pneumonia Hospitalar:

1. DIAGNOSTICO:
   - Criteros clinicos: febre > 38C, tosse produtiva, infiltrado radiologico
   - Tempo de aparecimento: >= 48h apos admissao hospitalar

2. EXAMES:
   - Hemoculturas (antes do antibiotico)
   - Cultura de e

Dataset de treino: 7712 exemplos
Dataset de validacao: 3306 exemplos


---
## 4. Configuracao do Modelo Base

### Escolha do Modelo

Para este projeto, utilizamos:
- **TinyLlama/TinyLlama-1.1B-Chat-v1.0** - Modelo leve e eficiente para fine-tuning
- Alternativas: Mistral-7B, LLaMA-2-7B, Falcon-7B

### Quantizacao 4-bit (QLoRA)
Reduz o consumo de memoria permitindo treinamento em GPUs com menor VRAM.

In [4]:
# Modelo base - TinyLlama para demo (trocar por modelo maior em producao)
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configuracao de quantizacao 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Carregar tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Carregar modelo com quantizacao
print("Carregando modelo base...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

print(f"Modelo {MODEL_NAME} carregado com sucesso!")
print(f"Parametros totais: {model.num_parameters():,}")

Carregando modelo base...


Loading weights: 100%|██████████| 201/201 [00:04<00:00, 49.26it/s]


Modelo TinyLlama/TinyLlama-1.1B-Chat-v1.0 carregado com sucesso!
Parametros totais: 1,100,048,384


---
## 5. Configuracao do LoRA

LoRA (Low-Rank Adaptation) permite treinar apenas uma fracao dos parametros do modelo, tornando o fine-tuning muito mais eficiente.

In [5]:
# Configuracao LoRA
lora_config = LoraConfig(
    r=16,                          # Rank de baixa dimensao
    lora_alpha=32,                 # Escala dos pesos LoRA
    target_modules=[               # Modulos alvo para adaptacao
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("LoRA configurado! O SFTTrainer aplicara automaticamente.")

LoRA configurado! O SFTTrainer aplicara automaticamente.


---
## 6. Configuracao do Treinamento

### Configuracao Conservadora para MPS (evitar crash)

| Parametro | Motivo |
|------------|--------|
| **max_steps=200** | Limita tempo total de treinamento |
| **batch_size=1** | Conservador para evitar crash de memoria |
| **gradient_accumulation=8** | Batch efetivo = 8 para estabilidade |
| **Sem bf16/fp16** | MPS tem instabilidade com mixed precision |
| **Sem gradient_checkpointing** | Causa instabilidade no SFTTrainer |
| **dataloader_num_workers=0** | macOS tem problemas com fork |

**Por que essas restricoes?** O MPS (Apple Silicon) tem memoria compartilhada e instabilidades com mixed precision. Configuracao conservadora evita crashes.

In [6]:
# Configuracao de treinamento para demo rapida (~30 min)
# Configuracao conservadora para evitar crash de memoria no MPS

training_args = TrainingArguments(
    output_dir="../models/assistente_medico_lora",
    
    # === CONFIGURACAO PARA DEMO RAPIDA ===
    num_train_epochs=1,                    # 1 epoca
    max_steps=200,                         # Limitar a 200 steps
    per_device_train_batch_size=1,         # Batch pequeno para MPS
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,         # Batch efetivo = 6
    
    # === SEM OTIMIZACOES AGRESSIVAS ===
    bf16=False,                            # Desabilitado - instavel no MPS
    fp16=False,                            # Desabilitado - nao suportado em MPS
    gradient_checkpointing=False,          # Desabilitado - causa instabilidade
    dataloader_num_workers=0,              # macOS tem problemas com fork
    
    # === HIPERPARAMETROS ===
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    
    # === LOGGING ===
    logging_steps=10,
    eval_strategy="no",                    # Sem avaliacao durante treino
    save_strategy="no",                    # Sem save durante treino
    report_to="none",
    remove_unused_columns=False,
)

print("Configuracao conservadora definida!")
print(f"Max steps: {training_args.max_steps}")
print(f"Batch efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

Configuracao conservadora definida!
Max steps: 200
Batch efetivo: 8


---
## 7. Treinamento do Modelo

O treinamento utiliza o SFTTrainer (Supervised Fine-Tuning Trainer) do Hugging Face TRL.

In [7]:
# Verificar colunas do dataset
print(f"Colunas disponiveis: {dataset['train'].column_names}")
print(f"\nExemplo do dataset:")
print(dataset['train'][0])

Colunas disponiveis: ['condition', 'instruction', 'input', 'output', 'source', 'id', 'text']

Exemplo do dataset:
{'condition': None, 'instruction': 'Voce e um assistente medico especializado em pesquisa clinica. Responda a pergunta de pesquisa com base no contexto fornecido. Cite as evidencias sempre que possivel.', 'input': "Contexto: To assess the results of transsphenoidal pituitary surgery in patients with Cushing's disease over a period of 18 years, and to determine if there are factors which will predict the outcome. Sixty-nine sequential patients treated surgically by a single surgeon in Newcastle upon Tyne between 1980 and 1997 were identified and data from 61 of these have been analysed. Retrospective analysis of outcome measures. Patients were divided into three groups (remission, failure and relapse) depending on the late outcome of their treatment as determined at the time of analysis, i.e. 88 months (median) years after surgery. Remission is defined as biochemical reversa

In [8]:
# Configurar o trainer com dataset completo
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=lora_config
)

print("Trainer configurado!")
print(f"Iniciando treinamento com {len(dataset['train'])} exemplos...")
print(f"Max steps: {training_args.max_steps}")

Truncating train dataset: 100%|██████████| 7712/7712 [00:00<00:00, 17676.68 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 7712/7712 [00:00<00:00, 78946.02 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 3306/3306 [00:00<00:00, 83815.09 examples/s]


Trainer configurado!
Iniciando treinamento com 7712 exemplos...
Max steps: 200


In [9]:
# Iniciar treinamento
# IMPORTANTE: Em um ambiente real, este processo pode levar varias horas
# Para demo, podemos executar poucas epocas ou usar dados reduzidos

print("="*50)
print("INICIANDO TREINAMENTO")
print("="*50)

try:
    train_result = trainer.train()
    
    print("\n" + "="*50)
    print("TREINAMENTO CONCLUIDO!")
    print("="*50)
    
    # Metricas do treinamento
    metrics = train_result.metrics
    print(f"\nMetricas:")
    print(f"  Loss final: {metrics['train_loss']:.4f}")
    print(f"  Tempo total: {metrics['train_runtime']:.0f} segundos")
    
except Exception as e:
    print(f"Erro durante treinamento: {e}")
    print("Continuando com demonstracao...")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/Users/rodrigofranco/Environment/FIAP/fase 3/aulas/projeto-assistente-medico/venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


INICIANDO TREINAMENTO


/Users/rodrigofranco/Environment/FIAP/fase 3/aulas/projeto-assistente-medico/venv/lib/python3.14/site-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
W0914 22:14:14.750000 72302 torch/_dynamo/convert_frame.py:2008] [3/256] torch._dynamo hit config.accumulated_recompile_limit (256)
W0914 22:14:14.750000 72302 torch/_dynamo/convert_frame.py:2008] [3/256]    function: '_dequantize_4bit_compute' (/Users/rodrigofranco/Environment/FIAP/fase 3/aulas/projeto-assistente-medico/venv/lib/python3.14/site-packages/bitsandbytes/backends/default/ops.py:262)
W0914 22:14:14.750000 72302 torch/_dyna

Step,Training Loss
10,2.012638
20,1.425344
30,1.156398
40,1.030766
50,0.946841
60,0.939695
70,0.803694
80,0.914480
90,0.850558
100,0.877059



TREINAMENTO CONCLUIDO!

Metricas:
  Loss final: 0.9872
  Tempo total: 10568 segundos


---
## 8. Avaliacao do Modelo

In [10]:
# Avaliacao do modelo
print("Avaliando modelo...")

eval_results = trainer.evaluate()
print(f"\nResultados da avaliacao:")
print(f"  Eval Loss: {eval_results['eval_loss']:.4f}")

# Perplexidade
import math
perplexity = math.exp(eval_results['eval_loss'])
print(f"  Perplexidade: {perplexity:.2f}")

Avaliando modelo...


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
0.961265,0.809976,200,0.816423,536687.000000,0.806648



Resultados da avaliacao:
  Eval Loss: 0.8100
  Perplexidade: 2.25


---
## 9. Teste do Modelo Fine-Tuned

In [11]:
# Funcao para gerar respostas
def gerar_resposta(prompt_text, max_new_tokens=256):
    """Gera resposta usando o modelo fine-tuned."""
    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return resposta

# Testes
testes = [
    "### Instruction:\nVoce e um assistente medico especializado no hospital. Responda sobre Pneumonia Hospitalar.\n\n### Input:\nQual o protocolo para Pneumonia Hospitalar?\n\n### Response:\n",
    "### Instruction:\nVoce e um assistente medico especializado em Terapia Intensiva.\n\n### Input:\nQuando devemos suspeitar de sepse em um paciente internado?\n\n### Response:\n"
]

for i, teste in enumerate(testes):
    print(f"\n{'='*50}")
    print(f"TESTE {i+1}")
    print(f"{'='*50}")
    resposta = gerar_resposta(teste)
    # Extrair apenas a resposta (apos ### Response:)
    if "### Response:" in resposta:
        resposta_final = resposta.split("### Response:")[1].strip()
    else:
        resposta_final = resposta
    print(f"Resposta:\n{resposta_final}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TESTE 1


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Resposta:
How is pneumonia hospitalar defined? Pneumonia hospitalar is a condition in which the infection spreads into the lungs and affects the ability of the lungs to function. It is often characterized by a high fever (101°C or higher), shortness of breath, and shortness of breath upon exertion. Pneumonia hospitalar is an acute respiratory illness that can occur in children and adults. It can be caused by a bacteria or a virus, but in most cases it is caused by a bacteria called Staphylococcus aureus. Pneumonia hospitalar is often associated with other illnesses, such as pneumonia, tuberculosis, and HIV/AIDS. The diagnosis of pneumonia hospitalar is usually based on the signs and symptoms present. The World Health Organization (WHO) recommends that all suspected pneumonia cases should be investigated by a health care provider. This may include a chest x-ray, a culture of the sputum, or the use of a bronchoscope to examine the lungs.

TESTE 2
Resposta:
How to diagnose and treat sepsi

---
## 10. Salvamento e Exportacao do Modelo

In [12]:
# Criar diretorio de saida
output_dir = "../models/assistente_medico_final"
os.makedirs(output_dir, exist_ok=True)

# Salvar modelo LoRA
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Modelo salvo em: {output_dir}")

# Salvar metadados do treinamento
metadados_treinamento = {
    "modelo_base": MODEL_NAME,
    "tecnica": "QLoRA",
    "lora_rank": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "batch_size": training_args.per_device_train_batch_size,
    "total_exemplos": len(dataset['train']),
    "eval_loss": eval_results['eval_loss'],
    "perplexidade": perplexity
}

with open(os.path.join(output_dir, "metadados_treinamento.json"), 'w') as f:
    json.dump(metadados_treinamento, f, indent=2)

print(f"\nMetadados salvos:")
print(json.dumps(metadados_treinamento, indent=2))

Modelo salvo em: ../models/assistente_medico_final

Metadados salvos:
{
  "modelo_base": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "tecnica": "QLoRA",
  "lora_rank": 16,
  "lora_alpha": 32,
  "epochs": 1,
  "learning_rate": 0.0002,
  "batch_size": 1,
  "total_exemplos": 7712,
  "eval_loss": 0.8099755048751831,
  "perplexidade": 2.2478529245641403
}


---
## 11. Resumo

### O que foi feito neste notebook:
1. Configuracao do modelo base com quantizacao 4-bit (QLoRA)
2. Aplicacao de LoRA para treinamento eficiente
3. Treinamento do modelo com dados medicos
4. Avaliacao do modelo (loss, perplexidade)
5. Teste com perguntas medicas
6. Salvamento do modelo fine-tuned

### Metricas importantes:
- **Loss**: Erro do modelo (menor e melhor)
- **Perplexidade**: Incerteza do modelo (menor e melhor)
- **Parametros treinaveis**: Apenas ~1% do total (eficiencia do LoRA)

### Proximo notebook:
O notebook `03_langchain_fundamentos.ipynb` apresentara os conceitos do LangChain para integrar o modelo com chains, prompts e agents.

In [13]:
print("\n=== NOTEBOOK 02 CONCLUIDO ===")
print("Modelo fine-tuned salvo e pronto para uso!")


=== NOTEBOOK 02 CONCLUIDO ===
Modelo fine-tuned salvo e pronto para uso!
